# 16. Warp Block SharedMemory Basics | Warp、Block 与 Shared Memory 基础

**难度：** Medium | **环境：** GPU optional | **标签：** `算子编程`, `CUDA`, `Shared Memory` | **目标人群：** 编译与系统入门者

![Block 内共享内存复用链](../docs/public/01_Hardware_Math_and_Systems/16_block_shared_memory_reuse.svg)

---

## 本节导读

同一数据被 Block 内多个线程重复使用时，反复访问 HBM 会产生不必要的外部流量。共享内存把这份数据保留在 Block 内，但同时受到容量和同步正确性的约束。

本节计算尾 Warp 利用率、复用前后的 HBM 字节量，并用容量、同步和流量收益检查共享内存候选。它主要支撑算子优化路线，也为 Triton Tile、融合和片上归约提供访存基础。

**关键词：** `CUDA`, `Warp`, `Shared Memory`

---

## 前置阅读

**导语：** 先看执行层级，再看 shared memory 分工和片上复用会更顺。

- [Group 1B: Single-Device Execution, Memory and Data Access | 1B: 单设备执行、内存与访存](./1B.md)
- [Group 1D: Execution Models, Asynchrony and Operator Mapping | 1D: 执行模型、异步调度与算子映射](./1D.md)
- [15. CUDA Execution Model | CUDA 执行模型](./15_CUDA_Execution_Model.ipynb)


## Q1：warp、block 和 thread 分别负责什么？


这三个概念处在不同层级：

- **Thread**：最小执行单元，负责执行最细粒度的工作。
- **Warp**：硬件实际调度的基本执行单元，通常 32 个线程一起执行相同指令。
- **Block**：程序组织与资源分配的基本单元，适合做协作、同步和局部复用。

理解它们的关键，不是死记名词，而是知道每一层解决什么问题。Thread 负责计算细节，Warp 负责并行执行，Block 负责协作边界。

```mermaid
flowchart TD
    T[Thread] --> W[Warp]
    W --> B[Block]
    B --> H[HBM / Global Memory]
    B --> S[Shared Memory]
    W --- C[SIMT]
    B --- R[Block-level cooperation]
```


### Q1 覆盖率计算：尾 Warp 有多少线程真正工作

把执行层和组织层区分开，再看 kernel 代码就不会混。

In [ ]:
def execution_coverage(num_threads, warp_size=32):
    """计算 Warp 数、尾 Warp 有效线程和线程槽利用率。"""
    if num_threads <= 0 or warp_size <= 0:
        raise ValueError('num_threads and warp_size must be positive')
    warps = (num_threads + warp_size - 1) // warp_size
    thread_slots = warps * warp_size
    last_warp_active_threads = num_threads - (warps - 1) * warp_size
    return {
        'threads': num_threads,
        'warps': warps,
        'last_warp_active_threads': last_warp_active_threads,
        'thread_slot_utilization': num_threads / thread_slots,
    }

for threads in [32, 48, 65]:
    print(execution_coverage(threads))
assert execution_coverage(48)['last_warp_active_threads'] == 16


## Q2：为什么 shared memory 能显著改善块内复用？


shared memory 位于 HBM 和寄存器之间，速度比 HBM 快得多，且适合 block 内线程共享。

它的价值在于：
- 避免同一批数据反复从 HBM 读取；
- 让 block 内多个线程复用同一份中间结果；
- 把中间状态保留在离计算更近的层级。

所以 shared memory 不是“更大的缓存”，而是“更适合 block 协作的局部高速缓冲”。很多 kernel 优化的收益，本质上都来自把数据搬到这里后减少了 HBM 访问。

```mermaid
flowchart LR
    H[HBM] --> S[Shared Memory]
    S --> R[Registers]
    B[Block 内线程] --> S
    B -. fewer reads .-> H
```


### Q2 对照验证：共享内存方案是否真正减少 HBM 流量

同一份中间数据如果会被多次用到，就更适合放在 shared memory。

In [ ]:
def shared_reuse_gain(tile_bytes, reuse_count, blocks):
    """返回基线和共享内存方案的 HBM 字节量及复用收益。"""
    if tile_bytes <= 0 or reuse_count <= 0 or blocks <= 0:
        raise ValueError('tile_bytes, reuse_count and blocks must be positive')
    baseline_hbm_bytes = tile_bytes * reuse_count * blocks
    shared_hbm_bytes = tile_bytes * blocks
    return {
        'baseline_hbm_bytes': baseline_hbm_bytes,
        'shared_hbm_bytes': shared_hbm_bytes,
        'saved_hbm_bytes': baseline_hbm_bytes - shared_hbm_bytes,
        'reuse_efficiency': 1 - shared_hbm_bytes / baseline_hbm_bytes,
    }

for reuse_count in [1, 2, 4]:
    print('reuse', reuse_count, '->', shared_reuse_gain(8192, reuse_count, 8))
assert shared_reuse_gain(8192, 1, 8)['saved_hbm_bytes'] == 0


## Q3：为什么后面学 Triton 时，shared memory 总会反复出现？


Triton 的很多优化都围绕 block 级数据组织展开，而 block 级组织离不开 shared memory。

当你在 Triton 里做 tile、mask、fusion 或 block-wise reduction 时，本质上都是在决定：
- 哪些数据先放到片上；
- 哪些数据在 block 内共享；
- 哪些中间结果不必回到 HBM。

所以 shared memory 不是一个单独知识点，而是后续块级编程的核心基础。先把这层讲通，后面的 Triton kernel 就不会像一堆抽象参数。

```mermaid
flowchart TD
    A[Block-level data organization] --> B[tile]
    A --> C[shared memory]
    A --> D[mask / boundary handling]
    B --> E[Triton kernel]
    C --> E
    D --> E
```


### Q3 硬门槛：复用收益能否通过容量与同步检查

先问自己：这里是不是在减少 HBM 访问、增加 block 内复用。

In [ ]:
def shared_memory_candidate(tile_bytes, reuse_count, blocks, shared_limit_bytes, synchronization_valid):
    """同时检查共享内存容量、同步正确性和可观测 HBM 收益。"""
    traffic = shared_reuse_gain(tile_bytes, reuse_count, blocks)
    reasons = []
    if tile_bytes > shared_limit_bytes:
        reasons.append('shared_capacity')
    if not synchronization_valid:
        reasons.append('synchronization')
    if traffic['saved_hbm_bytes'] <= 0:
        reasons.append('no_reuse_gain')
    return {'eligible': not reasons, 'reasons': reasons, **traffic}

good = shared_memory_candidate(32 * 1024, 4, 8, 64 * 1024, True)
too_large = shared_memory_candidate(96 * 1024, 4, 8, 64 * 1024, True)
no_reuse = shared_memory_candidate(32 * 1024, 1, 8, 64 * 1024, True)
assert good['eligible'] is True
assert too_large['eligible'] is False
assert no_reuse['eligible'] is False
print(good)
print(too_large)
print(no_reuse)


## ⚠️ 常见误区

- warp 不是 block，warp 是执行粒度，block 是组织粒度。
- shared memory 不是越大越好，关键在于 block 内复用是否足够。
- 看到 shared memory 不一定就是“更复杂”，很多时候只是把数据放到了更合适的位置。
- 先懂层级关系，再看 kernel 代码，理解会稳很多。

## 相关阅读

**导语：** 把 warp / block / shared memory 直接接到 Triton kernel 和归约实现上，更容易看清 tile 是怎么真正落地的。

- [02. Triton 算子开发：融合门控激活函数 (Fused SwiGLU)](../03_Triton_Kernels/02_Triton_Fused_SwiGLU.ipynb)
- [03. Triton 算子开发实战：Fused RMSNorm](../03_Triton_Kernels/03_Triton_Fused_RMSNorm.ipynb)
- [06. Triton 进阶：跨线程归约与数值稳定 (Safe Softmax)](../03_Triton_Kernels/06_Triton_Fused_Softmax.ipynb)
---